# Silver Layer ETL Pipeline

**Purpose:** Transform and load cleaned data from bronze to silver layer tables in the DataWarehouse catalog.

**Prerequisites:**
1. Bronze tables must be loaded with data
2. Silver tables must exist (run ddl_script.sql first)
3. DataWarehouse catalog and bronze/silver schemas must exist

**Tables Transformed:**
- `silver.crm_cust_info` - Deduplicated customers with standardized gender/marital status
- `silver.crm_prd_info` - Products with category extraction and end date calculation
- `silver.crm_sales_details` - Sales with date conversion and price validation
- `silver.erp_cust_az12` - ERP customers with ID normalization and date validation
- `silver.erp_loc_a101` - ERP locations with country name standardization
- `silver.erp_px_cat_g1v1` - ERP product categories (passthrough)

**Process:** Truncate and load pattern with data quality transformations, logging, and timing.

In [0]:
import time
from datetime import datetime
from pyspark.sql import SparkSession

# Configuration
CATALOG_NAME = "DataWarehouse"
SOURCE_SCHEMA = "bronze"
TARGET_SCHEMA = "silver"

# Table mappings: table_name -> display_name
TABLE_CONFIG = {
    # CRM Tables
    "crm_cust_info": "CRM Customer Info",
    "crm_prd_info": "CRM Product Info",
    "crm_sales_details": "CRM Sales Details",
    # ERP Tables
    "erp_cust_az12": "ERP Customer AZ12",
    "erp_loc_a101": "ERP Location A101",
    "erp_px_cat_g1v1": "ERP Product Category G1V1",
}

print("✓ Configuration loaded")
print(f"  Catalog: {CATALOG_NAME}")
print(f"  Source Schema: {SOURCE_SCHEMA}")
print(f"  Target Schema: {TARGET_SCHEMA}")
print(f"  Tables to transform: {len(TABLE_CONFIG)}")

✓ Configuration loaded
  Catalog: DataWarehouse
  Source Schema: bronze
  Target Schema: silver
  Tables to transform: 6


In [0]:
%sql
USE CATALOG DataWarehouse;

In [0]:
def load_silver_table(table_name, display_name, transformation_sql):
    """
    Load data from bronze to silver table using truncate-and-load with transformation.
    
    Args:
        table_name: Name of the table in both schemas (without schema prefix)
        display_name: Human-readable name for logging
        transformation_sql: Complete SQL with TRUNCATE + INSERT (semicolon-separated)
    
    Returns:
        Tuple of (success: bool, duration: float, rows_loaded: int, error_message: str)
    """
    start_time = time.time()
    source_table = f"{CATALOG_NAME}.{SOURCE_SCHEMA}.{table_name}"
    target_table = f"{CATALOG_NAME}.{TARGET_SCHEMA}.{table_name}"
    
    try:
        # Print status
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Transforming {source_table} → {target_table}...")
        
        # Split multi-statement SQL and execute each statement separately
        statements = [stmt.strip() for stmt in transformation_sql.split(';') if stmt.strip()]
        
        for stmt in statements:
            spark.sql(stmt)
        
        # Count rows loaded
        rows_loaded = spark.sql(f"SELECT COUNT(*) as cnt FROM {target_table}").first()[0]
        
        duration = time.time() - start_time
        print(f"✓ Loaded {rows_loaded:,} rows into {display_name} in {duration:.2f} seconds")
        
        return (True, duration, rows_loaded, None)
        
    except Exception as e:
        duration = time.time() - start_time
        error_msg = f"✗ ERROR transforming {display_name}: \n{str(e)}"
        print(error_msg)
        return (False, duration, 0, str(e))

def print_separator():
    """Print a simple separator line."""
    print("-" * 80)

print("✓ Helper functions defined")

✓ Helper functions defined


In [0]:
print("=" * 80)
print("LOADING CRM TABLES")
print("=" * 80)

crm_start_time = time.time()
crm_results = []

# ============================================================================
# CRM Customer Info - Deduplication and standardization
# ============================================================================
crm_cust_sql = """
TRUNCATE TABLE silver.crm_cust_info;

INSERT INTO silver.crm_cust_info(
    cst_id, cst_key, cst_firstname, cst_lastname, 
    cst_gndr, cst_marital_status, cst_create_date
)
SELECT 
    cst_id, 
    cst_key, 
    TRIM(cst_firstname) as cst_firstname,
    TRIM(cst_lastname) as cst_lastname,
    CASE 
        WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
        WHEN UPPER(TRIM(cst_gndr)) = 'M' THEN 'Male'
        ELSE 'n/a' 
    END as cst_gndr,
    CASE 
        WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
        WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
        ELSE 'n/a' 
    END as cst_marital_status,
    cst_create_date
FROM (
    SELECT 
        *, 
        ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag
    FROM bronze.crm_cust_info 
    WHERE cst_id IS NOT NULL
) as ranked_cst
WHERE flag = 1
"""

result = load_silver_table("crm_cust_info", "CRM Customer Info", crm_cust_sql)
crm_results.append(("crm_cust_info", result))
print_separator()

# ============================================================================
# CRM Product Info - Category extraction and end date calculation
# ============================================================================
crm_prd_sql = """
TRUNCATE TABLE silver.crm_prd_info;

INSERT INTO silver.crm_prd_info (
    prd_id, cat_id, prd_key, prd_nm, prd_cost, 
    prd_line, prd_start_dt, prd_end_dt
)
SELECT 
    prd_id,
    REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,
    SUBSTRING(prd_key, 7, LENGTH(prd_key)) AS prd_key,
    prd_nm,
    COALESCE(prd_cost, 0) as prd_cost,
    CASE UPPER(TRIM(prd_line)) 
        WHEN 'M' THEN 'Mountain'
        WHEN 'R' THEN 'Road'
        WHEN 'S' THEN 'Other Sales'
        WHEN 'T' THEN 'Touring'
        ELSE 'n/a' 
    END AS prd_line,
    CAST(prd_start_dt AS DATE) as prd_start_dt,
    CAST(
        DATE_ADD(
            LEAD(prd_start_dt) OVER (PARTITION BY SUBSTRING(prd_key, 7, LENGTH(prd_key)) ORDER BY prd_start_dt), 
            -1
        ) AS DATE
    ) as prd_end_dt
FROM bronze.crm_prd_info
"""

result = load_silver_table("crm_prd_info", "CRM Product Info", crm_prd_sql)
crm_results.append(("crm_prd_info", result))
print_separator()

# ============================================================================
# CRM Sales Details - Date conversion and price validation
# ============================================================================
crm_sales_sql = """
TRUNCATE TABLE silver.crm_sales_details;

INSERT INTO silver.crm_sales_details (
    sls_ord_num, sls_prd_key, sls_cust_id, 
    sls_order_dt, sls_ship_dt, sls_due_dt, 
    sls_sales, sls_quantity, sls_price
)
SELECT 
    sls_ord_num, 
    sls_prd_key, 
    sls_cust_id,
    CASE 
        WHEN sls_order_dt = 0 OR LENGTH(CAST(sls_order_dt AS STRING)) != 8 THEN NULL
        ELSE TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd') 
    END as sls_order_dt,
    CASE 
        WHEN sls_ship_dt = 0 OR LENGTH(CAST(sls_ship_dt AS STRING)) != 8 THEN NULL
        ELSE TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') 
    END as sls_ship_dt,
    CASE 
        WHEN sls_due_dt = 0 OR LENGTH(CAST(sls_due_dt AS STRING)) != 8 THEN NULL
        ELSE TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd') 
    END as sls_due_dt,
    CASE 
        WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price)
        THEN sls_quantity * ABS(sls_price)
        ELSE sls_sales 
    END as sls_sales,
    sls_quantity,
    CASE 
        WHEN sls_price IS NULL OR sls_price <= 0
        THEN sls_sales / NULLIF(sls_quantity, 0)
        ELSE sls_price 
    END AS sls_price
FROM bronze.crm_sales_details
"""

result = load_silver_table("crm_sales_details", "CRM Sales Details", crm_sales_sql)
crm_results.append(("crm_sales_details", result))
print_separator()

crm_duration = time.time() - crm_start_time
crm_success_count = sum(1 for _, (success, _, _, _) in crm_results if success)

print(f"\n✓ CRM Tables Complete: {crm_success_count}/{len(crm_results)} successful in {crm_duration:.2f} seconds\n")

LOADING CRM TABLES
[2026-05-06 07:46:43] Transforming DataWarehouse.bronze.crm_cust_info → DataWarehouse.silver.crm_cust_info...
✓ Loaded 18,484 rows into CRM Customer Info in 4.33 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:46:47] Transforming DataWarehouse.bronze.crm_prd_info → DataWarehouse.silver.crm_prd_info...
✓ Loaded 397 rows into CRM Product Info in 3.39 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:46:50] Transforming DataWarehouse.bronze.crm_sales_details → DataWarehouse.silver.crm_sales_details...
✓ Loaded 60,398 rows into CRM Sales Details in 3.05 seconds
--------------------------------------------------------------------------------

✓ CRM Tables Complete: 3/3 successful in 10.77 seconds



In [0]:
print("=" * 80)
print("LOADING ERP TABLES")
print("=" * 80)

erp_start_time = time.time()
erp_results = []

# ============================================================================
# ERP Customer AZ12 - ID normalization and date validation
# ============================================================================
erp_cust_sql = """
TRUNCATE TABLE silver.erp_cust_az12;

INSERT INTO silver.erp_cust_az12(cid, bdate, gen)
SELECT 
    CASE 
        WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LENGTH(cid)) 
        ELSE cid 
    END AS cid,
    CASE 
        WHEN bdate > CURRENT_DATE() THEN NULL 
        ELSE bdate 
    END as bdate,
    CASE 
        WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
        WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
        ELSE 'n/a' 
    END AS gen
FROM bronze.erp_cust_az12
"""

result = load_silver_table("erp_cust_az12", "ERP Customer AZ12", erp_cust_sql)
erp_results.append(("erp_cust_az12", result))
print_separator()

# ============================================================================
# ERP Location A101 - Country name standardization
# ============================================================================
erp_loc_sql = """
TRUNCATE TABLE silver.erp_loc_a101;

INSERT INTO silver.erp_loc_a101(cid, cntry)
SELECT 
    REPLACE(cid, '-', '') as cid,
    CASE 
        WHEN TRIM(cntry) = 'DE' THEN 'Germany'
        WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
        WHEN TRIM(cntry) = '' OR TRIM(cntry) IS NULL THEN 'n/a'
        ELSE TRIM(cntry) 
    END AS cntry
FROM bronze.erp_loc_a101
"""

result = load_silver_table("erp_loc_a101", "ERP Location A101", erp_loc_sql)
erp_results.append(("erp_loc_a101", result))
print_separator()

# ============================================================================
# ERP Product Category G1V1 - Passthrough (no transformation)
# ============================================================================
erp_cat_sql = """
TRUNCATE TABLE silver.erp_px_cat_g1v1;

INSERT INTO silver.erp_px_cat_g1v1(id, cat, subcat, maintenance)
SELECT id, cat, subcat, maintenance 
FROM bronze.erp_px_cat_g1v1
"""

result = load_silver_table("erp_px_cat_g1v1", "ERP Product Category G1V1", erp_cat_sql)
erp_results.append(("erp_px_cat_g1v1", result))
print_separator()

erp_duration = time.time() - erp_start_time
erp_success_count = sum(1 for _, (success, _, _, _) in erp_results if success)

print(f"\n✓ ERP Tables Complete: {erp_success_count}/{len(erp_results)} successful in {erp_duration:.2f} seconds\n")

LOADING ERP TABLES
[2026-05-06 07:47:12] Transforming DataWarehouse.bronze.erp_cust_az12 → DataWarehouse.silver.erp_cust_az12...
✓ Loaded 18,484 rows into ERP Customer AZ12 in 3.82 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:47:15] Transforming DataWarehouse.bronze.erp_loc_a101 → DataWarehouse.silver.erp_loc_a101...
✓ Loaded 18,484 rows into ERP Location A101 in 2.59 seconds
--------------------------------------------------------------------------------
[2026-05-06 07:47:18] Transforming DataWarehouse.bronze.erp_px_cat_g1v1 → DataWarehouse.silver.erp_px_cat_g1v1...
✓ Loaded 37 rows into ERP Product Category G1V1 in 2.67 seconds
--------------------------------------------------------------------------------

✓ ERP Tables Complete: 3/3 successful in 9.08 seconds



In [0]:
print("=" * 80)
print("SILVER LAYER LOAD SUMMARY")
print("=" * 80)

# Combine all results
all_results = crm_results + erp_results
total_duration = crm_duration + erp_duration
total_success = crm_success_count + erp_success_count
total_tables = len(all_results)
total_rows = sum(rows for _, (success, _, rows, _) in all_results if success)

print(f"\nExecution Time: {total_duration:.2f} seconds")
print(f"Tables Processed: {total_tables}")
print(f"Successful: {total_success}")
print(f"Failed: {total_tables - total_success}")
print(f"Total Rows Loaded: {total_rows:,}")

# Detailed results
print("\nDetailed Results:")
for table_name, (success, duration, rows, error) in all_results:
    status = "✓ SUCCESS" if success else "✗ FAILED"
    row_info = f"{rows:>8,} rows" if success else "      0 rows"
    print(f"  {status} | {table_name:<25} | {row_info} | {duration:>6.2f}s")
    if error:
        print(f"           Error: {error}")

# Check for failures
if total_success < total_tables:
    print("\n⚠ WARNING: Some tables failed to load. Review errors above.")
    failed_tables = [name for name, (success, _, _, _) in all_results if not success]
    print(f"Failed tables: {', '.join(failed_tables)}")
else:
    print("\n✓ All silver tables loaded successfully!")
    print("\nData Quality Transformations Applied:")
    print("  • Customer deduplication (latest record by create date)")
    print("  • Gender/marital status standardization")
    print("  • Product category extraction from keys")
    print("  • Product end date calculation using LEAD window function")
    print("  • Sales date conversion from integer (yyyyMMdd format)")
    print("  • Sales price validation and recalculation")
    print("  • ERP customer ID normalization (removed 'NAS' prefix)")
    print("  • ERP country name standardization")
    print("  • Future date validation (set to NULL if > current date)")

print("=" * 80)

SILVER LAYER LOAD SUMMARY

Execution Time: 19.85 seconds
Tables Processed: 6
Successful: 6
Failed: 0
Total Rows Loaded: 116,284

Detailed Results:
  ✓ SUCCESS | crm_cust_info             |   18,484 rows |   4.33s
  ✓ SUCCESS | crm_prd_info              |      397 rows |   3.39s
  ✓ SUCCESS | crm_sales_details         |   60,398 rows |   3.05s
  ✓ SUCCESS | erp_cust_az12             |   18,484 rows |   3.82s
  ✓ SUCCESS | erp_loc_a101              |   18,484 rows |   2.59s
  ✓ SUCCESS | erp_px_cat_g1v1           |       37 rows |   2.67s

✓ All silver tables loaded successfully!

Data Quality Transformations Applied:
  • Customer deduplication (latest record by create date)
  • Gender/marital status standardization
  • Product category extraction from keys
  • Product end date calculation using LEAD window function
  • Sales date conversion from integer (yyyyMMdd format)
  • Sales price validation and recalculation
  • ERP customer ID normalization (removed 'NAS' prefix)
  • ERP country